In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType

spark = SparkSession.builder.appName("SuperstoreDataCleaning").getOrCreate()
df = spark.read.csv(
    "Sample - Superstore.csv", 
    header=True, 
    inferSchema=True, 
    quote='"', 
    escape='"'
)

df_q3 = df.dropDuplicates(["Customer ID", "Order Date"])


df_q4 = df.filter(F.col("Region") == "West") \
          .groupBy("Category") \
          .agg(F.avg("Sales").alias("avg_sales"))


df_q5 = df.na.fill({"Ship Mode": "Unknown"})

df_q6 = df.groupBy("City") \
          .count() \
          .filter(F.col("count") > 100)


df_q8 = df.filter(
    (F.col("Quantity") >= 2) & 
    (F.col("Quantity") <= 5) & 
    (F.col("Segment") == "Corporate")
)

df_q10 = df.withColumn("Order Timestamp", F.to_timestamp(F.col("Order Date"), "M/d/yyyy")) \
           .drop("Order Date")


df_q12 = df.filter(
    F.col("Customer ID").isNotNull() & 
    (F.col("Customer Name") != "") & 
    F.col("Customer Name").isNotNull()
)


df_q13 = df.agg(
    F.min("Sales").alias("min_sales"),
    F.max("Sales").alias("max_sales"),
    F.mean("Sales").alias("mean_sales")
)


final_pipeline_df = df.dropDuplicates() \
                      .na.fill({"Sales": 0.0}) \
                      .groupBy("Customer ID") \
                      .agg(F.sum("Sales").alias("total_revenue"))

In [ ]:

print("--- Final Pipeline Schema ---")
final_pipeline_df.printSchema()


print("--- Final Pipeline Data ---")
final_pipeline_df.show()

print("--- Top 10 Customers by Revenue ---")
final_pipeline_df.orderBy(F.col("total_revenue").desc()).show(10)


total_customers = final_pipeline_df.count()
print(f"Total unique customers processed: {total_customers}")

--- Final Pipeline Schema ---
root
 |-- Customer ID: string (nullable = true)
 |-- total_revenue: double (nullable = true)

--- Final Pipeline Data ---
+-----------+------------------+
|Customer ID|     total_revenue|
+-----------+------------------+
|   VW-21775|          6134.038|
|   RR-19315|           615.932|
|   PB-19210|           132.738|
|   MY-17380|          2254.285|
|   EM-13960|           933.704|
|   MS-17530|475.65599999999995|
|   KH-16630|3918.9660000000003|
|   SW-20275|1966.6499999999996|
|   BD-11500|          4411.243|
|   AH-10690| 7888.294000000001|
|   JF-15490|1082.9180000000001|
|   JF-15415|          2371.448|
|   PH-18790|           729.648|
|   IM-15070|          4930.474|
|   PW-19240|          3922.415|
|   NW-18400| 7234.014000000001|
|   JH-15985|          7954.998|
|   KM-16225|          2260.958|
|   KF-16285|10604.266000000001|
|   KD-16615|           2243.51|
+-----------+------------------+
only showing top 20 rows
--- Top 10 Customers by Revenue